# DaT Parkinson's Challenge — carnet Colab

Chaîne complète : prétraitement → modèles → fusion → `submission.zip`.

**À lire avant de lancer**

1. **Règlement.** Les données ne doivent pas être transmises à un service tiers ni à une API d'IA (ChatGPT, Gemini, etc.), et il faut empêcher toute personne n'ayant pas accepté le règlement d'y accéder. Colab est un environnement de calcul, pas une API d'IA, mais les données transitent quand même par des serveurs Google. Si tu as le moindre doute, pose la question sur le forum du challenge avant d'y téléverser quoi que ce soit. Dans tous les cas : **compte privé, pas de notebook partagé, pas de dossier Drive partagé.**
2. **Versions.** L'environnement d'évaluation est figé. Les artefacts produits ici doivent s'y recharger. La cellule 2 épingle les versions qui comptent.
3. **Sessions.** Colab coupe. Le cache et les poids vont sur Drive, et l'entraînement reprend pli par pli (`--resume 1`).

| GPU Colab | bf16 | Réglage conseillé |
|---|---|---|
| T4 (gratuit) | non | `--amp fp16 --batch-size 8` |
| L4 / A100 (Pro) | oui | `--amp auto --batch-size 16` |


## 1. Environnement

In [ ]:
import os, subprocess, shutil, sys
print(sys.version)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout or 'aucun GPU')
print(f"vCPU : {os.cpu_count()}")
print(subprocess.run(['free','-g'], capture_output=True, text=True).stdout.splitlines()[1])
print(shutil.disk_usage('/content'))

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/dat-parkinsons'   # survit aux deconnexions
os.makedirs(WORK, exist_ok=True)
print('travail persistant :', WORK)

## 2. Dépendances épinglées

Ces versions sont celles du `uv.lock` du dépôt runtime officiel. Deux formats d'artefact sont sensibles à la version :

- le **pipeline scikit-learn** est sérialisé par pickle — une version différente peut refuser de se recharger dans le conteneur ;
- l'architecture **timm** de `proj2d` doit être constructible côté conteneur.

LightGBM est sauvegardé en texte (`model_to_string`), donc portable entre versions — c'est voulu.

Colab demandera sans doute un redémarrage du noyau après cette cellule : accepte, puis reprends à la cellule 3.

In [ ]:
!pip install -q "scikit-learn==1.8.0" "lightgbm==4.6.0" "nibabel==5.4.2" \
               "timm==1.0.27" "scikit-image==0.26.0"
import sklearn, lightgbm, nibabel, torch, timm
print('sklearn', sklearn.__version__, '(conteneur 1.8.0)')
print('lightgbm', lightgbm.__version__, '(conteneur 4.6.0)')
print('nibabel', nibabel.__version__, '(conteneur 5.4.2)')
print('timm', timm.__version__, '(conteneur 1.0.27)')
print('torch', torch.__version__, '(conteneur 2.12.1+cu129)')
print('bf16 supporte :', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## 3. Code

Dépose `dat-parkinsons-pipeline.zip` sur ton Drive, ou téléverse-le ici.

In [ ]:
import os, zipfile, shutil

ZIP = f'{WORK}/dat-parkinsons-pipeline.zip'
if not os.path.exists(ZIP):
    from google.colab import files
    up = files.upload()
    ZIP = '/content/' + next(iter(up))

shutil.rmtree('/content/code', ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/code')
REPO = '/content/code/dat-parkinsons'
os.chdir(REPO)
!ls -R . | head -30

## 4. Données

Récupère les URL signées depuis la page *Data download* du challenge (tu dois être connecté ; elles expirent).

Colle-les ci-dessous. **Ne mets jamais tes identifiants DrivenData dans une cellule.**

L'archive brute est copiée sur Drive une fois pour toutes : les sessions suivantes repartent de là au lieu de retélécharger.

In [ ]:
import os, glob

NIFTI_URL  = ''   # <- URL signee de l'archive d'images
LABELS_URL = ''   # <- URL signee de train_labels.csv

os.makedirs('/content/data/niftis', exist_ok=True)
ARCHIVE = f'{WORK}/raw_niftis.tar'

if os.path.exists(ARCHIVE):
    print('archive trouvee sur Drive, extraction locale')
    !tar -xf "$ARCHIVE" -C /content/data
elif NIFTI_URL:
    !wget -q --show-progress -O /content/raw.archive "$NIFTI_URL"
    !mkdir -p /content/data && tar -xf /content/raw.archive -C /content/data \
        || unzip -q /content/raw.archive -d /content/data
    # sauvegarde pour les sessions suivantes (rapide a relire depuis Drive)
    !tar -cf "$ARCHIVE" -C /content/data niftis
else:
    raise SystemExit('renseigne NIFTI_URL, ou depose raw_niftis.tar sur Drive')

if LABELS_URL:
    !wget -q -O /content/data/train_labels.csv "$LABELS_URL"
    !cp /content/data/train_labels.csv "$WORK/train_labels.csv"
elif os.path.exists(f'{WORK}/train_labels.csv'):
    !cp "$WORK/train_labels.csv" /content/data/train_labels.csv

n = len(glob.glob('/content/data/niftis/*.nii.gz'))
print(f'{n} examens')
!head -3 /content/data/train_labels.csv

## 5. Auto-test du code réseau

Trente secondes. Il vérifie la convention d'axes de l'augmentation, les formes, les gradients et la capacité à sur-apprendre un mini-lot. Le code CNN n'a jamais tourné sur GPU avant ce point — ne lance rien de long tant que ce n'est pas vert.

In [ ]:
!python tools/selftest_cnn.py

## 6. Prétraitement

Environ 0,5 s par examen et par cœur. Le cache va sur Drive : il ne sera calculé qu'une fois, quelles que soient les déconnexions.

Regarde la sortie : nombre de pseudo-centres, examens `ok=False`, prévalence par centre. C'est le premier vrai contrôle qualité sur les données.

In [ ]:
import os
CACHE = '/content/cache'
os.makedirs(CACHE, exist_ok=True)
if os.path.exists(f'{WORK}/cache/volumes.npy'):
    print('cache trouve sur Drive, copie locale')
    !cp "$WORK/cache/volumes.npy" "$WORK/cache/manifest.csv" "$CACHE/"
else:
    !python scripts/01_preprocess.py --data /content/data --out "$CACHE" \
        --workers {os.cpu_count()}
    !mkdir -p "$WORK/cache" && cp "$CACHE/volumes.npy" "$CACHE/manifest.csv" "$WORK/cache/"

In [ ]:
import pandas as pd
man = pd.read_csv(f'{CACHE}/manifest.csv')
print(man.groupby('pseudo_center')['is_pathologic'].agg(['size','mean']).round(3))
print('\nechecs :', int((~man['ok'].astype(bool)).sum()))
print(man[['background','peak_ratio','yaw_deg','midline_shift_mm','slab_z_frac']]
      .describe().round(2))

### Contrôle visuel

À faire vraiment. Si le recadrage rate sur un centre, ça se voit immédiatement ici et nulle part ailleurs.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
vols = np.load(f'{CACHE}/volumes.npy', mmap_mode='r')
cent = man['pseudo_center'].unique()[:4]
fig, axes = plt.subplots(len(cent), 6, figsize=(15, 2.6*len(cent)))
axes = np.atleast_2d(axes)
for r, c in enumerate(cent):
    idx = man.index[man['pseudo_center'] == c][:6]
    for j, i in enumerate(idx):
        v = np.asarray(vols[i], dtype=np.float32)
        axes[r, j].imshow(v[:, :, 16:24].max(2).T, cmap='hot', origin='lower',
                          vmin=0, vmax=6)
        axes[r, j].set_title(f"y={man.loc[i,'is_pathologic']:.0f}", fontsize=8)
        axes[r, j].axis('off')
    axes[r, 0].set_ylabel(c[:14], fontsize=7)
plt.suptitle('MIP axiale du recadrage striatal — les deux noyaux doivent etre '
             'centres et symetriques', fontsize=10)
plt.tight_layout(); plt.show()

## 7. Modèles sur descripteurs — première soumission

Quelques secondes. Fais une vraie soumission **dès aujourd'hui** avec ça : elle valide le format de bout en bout et fixe ta référence, bien avant d'engager du temps GPU.

In [ ]:
!python scripts/02_train_tab.py --cache "$CACHE" --out /content/models/tab
!python scripts/04_blend.py --out /content/models/blend --manifest "$CACHE/manifest.csv" \
    --oof gbm=/content/models/tab/oof_gbm.csv \
    --oof linear=/content/models/tab/oof_linear.csv
!python scripts/05_pack.py --tab /content/models/tab --blend /content/models/blend \
    --out /content/dist
from google.colab import files
files.download('/content/dist/submission.zip')

## 8. CNN

`--resume 1` : chaque pli terminé est écrit sur Drive, la session suivante reprend où elle s'est arrêtée.

Commence par un test court (`--epochs 4`) pour mesurer le temps par pli, puis dimensionne. Sur T4, `--amp fp16 --batch-size 8`.

In [ ]:
import torch, os
BS   = 16 if torch.cuda.is_bf16_supported() else 8
AMP  = 'auto' if torch.cuda.is_bf16_supported() else 'fp16'
OUT3 = f'{WORK}/models/resnet3d'
os.makedirs(OUT3, exist_ok=True)
print(f'batch {BS} | amp {AMP}')

# calibrage du temps : 4 epoques, 1 pli
!python scripts/03_train_cnn.py --cache "$CACHE" --out /content/timing \
    --model resnet3d --epochs 4 --folds 5 --batch-size {BS} --amp {AMP} \
    --eval-every 4 --resume 0 2>&1 | head -20

In [ ]:
!python scripts/03_train_cnn.py --cache "$CACHE" --out "$OUT3" \
    --model resnet3d --epochs 60 --folds 5 --batch-size {BS} --amp {AMP} \
    --eval-every 5 --resume 1

In [ ]:
# Regler --epochs : relance avec la valeur au creux de cette courbe.
import pandas as pd, matplotlib.pyplot as plt
cur = pd.read_csv(f'{OUT3}/curve.csv').groupby('epoch')['val_logloss'].mean()
cur.plot(marker='o'); plt.ylabel('log loss validation'); plt.grid(alpha=.3); plt.show()
print('minimum a l\'epoque', int(cur.idxmin()), ':', round(float(cur.min()), 4))

### 2.5D sur encodeur pré-entraîné

`--pretrained 1` télécharge les poids ImageNet **maintenant** (Colab a le réseau) ; ils partent ensuite dans les `fold*.pt` du zip, puisque le conteneur d'évaluation n'a pas d'accès réseau.

Vérifie la licence du backbone : `convnext_tiny` et les `resnet` de timm sont permissifs, ce qui satisfait la clause *external data with rights* et la licence MIT imposée aux gagnants.

In [ ]:
OUT2 = f'{WORK}/models/proj2d'
!python scripts/03_train_cnn.py --cache "$CACHE" --out "$OUT2" \
    --model proj2d --backbone convnext_tiny --pretrained 1 \
    --epochs 30 --folds 5 --batch-size {max(BS,16)} --lr 1e-4 --amp {AMP} --resume 1

## 9. Fusion, calibration, archive

Suis **l'estimation croisée de la calibration**, pas la log loss brute : cette dernière est optimiste, puisque la fusion et le Platt sont ajustés sur les mêmes points hors-pli.

Regarde aussi le détail par pseudo-centre : un centre nettement moins bon que les autres est le meilleur indicateur de ce qui t'attend sur le test privé.

In [ ]:
!python scripts/04_blend.py --out /content/models/blend --manifest "$CACHE/manifest.csv" \
    --oof gbm=/content/models/tab/oof_gbm.csv \
    --oof linear=/content/models/tab/oof_linear.csv \
    --oof resnet3d="$OUT3/oof.csv" \
    --oof proj2d="$OUT2/oof.csv" \
    --shrink 0.05

In [ ]:
!python scripts/05_pack.py --tab /content/models/tab --blend /content/models/blend \
    --cnn resnet3d="$OUT3" --cnn proj2d="$OUT2" --out /content/dist
!cp /content/dist/submission.zip "$WORK/submission.zip"
from google.colab import files
files.download('/content/dist/submission.zip')

## 10. Répétition de l'inférence

Rejoue le scénario du conteneur sur 200 examens d'entraînement. Ça ne remplace pas `just test-submission` dans l'image officielle, mais ça attrape les erreurs d'import, de chemin et de format — et ça mesure le débit, à extrapoler au test complet contre la limite de 3 h.

In [ ]:
import os, shutil, zipfile, pandas as pd, numpy as np, time
shutil.rmtree('/content/ce', ignore_errors=True)
os.makedirs('/content/ce/data/niftis')
man = pd.read_csv(f'{CACHE}/manifest.csv')
sample = man.sample(200, random_state=0)
for u in sample['uid']:
    shutil.copy(f'/content/data/niftis/{u}.nii.gz', f'/content/ce/data/niftis/{u}.nii.gz')
sample.assign(is_pathologic=0.5)[['uid','is_pathologic']] \
      .to_csv('/content/ce/data/submission_format.csv', index=False)
with zipfile.ZipFile('/content/dist/submission.zip') as z:
    z.extractall('/content/ce')

t0 = time.time()
!cd /content/ce && DATA_DIR=/content/ce/data python main.py
dt = time.time() - t0

sub = pd.read_csv('/content/ce/submission.csv')
fmt = pd.read_csv('/content/ce/data/submission_format.csv')
print('\ncolonnes exactes :', list(sub.columns) == ['uid','is_pathologic'])
print('ordre identique  :', bool((sub.uid.values == fmt.uid.values).all()))
print('bornes (0,1)     :', bool(sub.is_pathologic.between(0,1).all()))
print('aucun NaN        :', bool(sub.is_pathologic.notna().all()))
print(f'debit : {dt/len(sub):.3f} s/examen -> {dt/len(sub)*3000/60:.0f} min pour 3000 examens '
      f'(limite 180 min)')

---

### Avant la soumission finale

- Passe d'abord par le **smoke test** sur le site (6 minutes max), jamais directement par la soumission complète.
- Teste dans **l'image officielle** si tu peux : `just test-submission` du dépôt runtime. Le carnet ne reproduit pas les versions exactes du conteneur.
- Choisis la soumission finale sur la robustesse **par pseudo-centre**, pas sur le tableau public. Le classement se joue sur un test privé de composition inconnue.
- Garde de la marge : la file d'exécution est partagée entre tous les concurrents.